In [1]:
import intake
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import geopandas as gpd

In [2]:
! pip install --upgrade xarray zarr gcsfs cftime nc-time-axis

In [3]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import gcsfs

xr.set_options(display_style='html')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [4]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
df['activity_id'].unique()

<StringArray>
[ 'HighResMIP',        'CMIP',       'CFMIP', 'ScenarioMIP',  'AerChemMIP',
       'RFMIP',      'FAFMIP',       'DAMIP',       'LUMIP',      'CDRMIP',
       'GMMIP',       'C4MIP',        'OMIP',        'PMIP',      'LS3MIP',
        'DCPP',       'PAMIP',      'ISMIP6']
Length: 18, dtype: str

In [5]:
pr_df = df[
    (df["activity_id"] == "CMIP") &
    (df["experiment_id"] == "historical") &
    (df["table_id"] == "Amon") &
    (df["variable_id"] == "pr") &
    (df["source_id"] == "GFDL-ESM4") &
    (df["member_id"] == "r1i1p1f1")
]

pr_df

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
244715,CMIP,NOAA-GFDL,GFDL-ESM4,historical,r1i1p1f1,Amon,pr,gr1,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/hist...,NaN,20190726


In [6]:
# this only needs to be created once
gcs = gcsfs.GCSFileSystem(token='anon')

# get the path to a specific zarr store (the first one from the dataframe above)
zstore = pr_df.loc[244715].zstore

# create a mutable-mapping-style interface to the store
mapper = gcs.get_mapper(zstore)

# open it using xarray and zarr
ds = xr.open_zarr(mapper, consolidated=True)
ds

I0602 14:40:20.284201 3919838 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0602 14:40:20.298350 3919860 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)


<xarray.Dataset> Size: 411MB
Dimensions:    (time: 1980, lat: 180, lon: 288, bnds: 2)
Coordinates:
  * time       (time) object 16kB 1850-01-16 12:00:00 ... 2014-12-16 12:00:00
  * lat        (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon        (lon) float64 2kB 0.625 1.875 3.125 4.375 ... 356.9 358.1 359.4
  * bnds       (bnds) float64 16B 1.0 2.0
    lat_bnds   (lat, bnds) float64 3kB ...
    lon_bnds   (lon, bnds) float64 5kB ...
    time_bnds  (time, bnds) object 32kB ...
Data variables:
    pr         (time, lat, lon) float32 411MB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.0 UGRID-1.0
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  36500.0
    comment:                <null ref>
    ...                     ...
    title:                  NOAA GFDL GFDL-ESM4 model output prepared for CMI...
    tracking_id:            hdl:21.14100/29468e1c-b66b-40d6-92aa-9851fba964ee...
    variable_id:            pr
    variant_info:           N/A
    variant_label:          r1i1p1f1
    status:                 2019-09-10;created;by nhn2@columbia.edu

In [7]:
df = ds.to_dataframe().reset_index()


In [8]:
month_names = {
    1: 'January',
    2: 'February',
    3: 'March',
    4: 'April',
    5: 'May',
    6: 'June',
    7: 'July',
    8: 'August',
    9: 'September',
    10: 'October',
    11: 'November',
    12: 'December'
}

df['month'] = df['time'].apply(lambda x: month_names[x.month])
df['year'] = df['time'].apply(lambda x: x.year)
df['day'] = df['time'].apply(lambda x: x.day)

In [9]:
df_1887 = df[df['year']==1887]
df_1887['pr']=df_1887['pr']*86400

In [10]:
def make_poly(row):
    return Polygon([
        (row["lon_bnds"][0], row["lat_bnds"][0]),
        (row["lon_bnds"][1], row["lat_bnds"][0]),
        (row["lon_bnds"][1], row["lat_bnds"][1]),
        (row["lon_bnds"][0], row["lat_bnds"][1])
    ])

In [11]:
gdf = gpd.GeoDataFrame(
    df_1887,
    geometry=gpd.points_from_xy(df_1887["lon"], df_1887["lat"]),
    crs="EPSG:4326"
)

In [12]:
china = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_1_states_provinces.zip"
)
china = china[china["admin"] == "China"]

In [13]:
joined = gpd.sjoin(gdf, china, predicate="intersects")
province_rain = joined.groupby(["name", "month"]).agg({
    "pr": ["sum", "mean", "max"]
}).reset_index()
province_rain.to_csv('data/province_pr.csv')


In [14]:
df_filtered = joined[joined["name"].isin(["Hubei", "Hunan", "Jianxi",'Anhui','Jiangsu'])]

province_rain = df_filtered.groupby(["name", "month"]).agg({
    "pr": ["sum", "mean", "max"]
}).reset_index()

province_rain.columns = ["name", "month", "pr_sum", "pr_mean", "pr_max"]


# province_rain.columns = ["name", "pr_sum", "pr_mean", "pr_max"]
choropleth = china.merge(province_rain, on="name")
choropleth.to_file(
    "yangtze.geojson",
    driver="GeoJSON"
)

In [15]:
choro = pd.DataFrame(choropleth.drop(columns='geometry'))
choro.to_json('data/yangtze.json',orient='records',indent=4)